In [ ]:
import os 
import pandas as pd
from Bio import SeqIO
import re

In [ ]:
def limit_length(bed_file, Min_Length=50, Max_Length=10000):
    df_bed = pd.read_csv(bed_file, sep="\t")
    df_bed["Length"] = df_bed["end"] - df_bed["start"] + 1
    df_limit_bed = df_bed[(Min_Length <= df_bed["Length"]) & (df_bed["Length"] <= Max_Length)]
    return df_limit_bed

In [ ]:
def limit_split_disc(df_limit_bed, split_num=1, disc_num=1):
    df_limit_sd = df_limit_bed[(df_limit_bed["discordants"] >= disc_num) & (df_limit_bed["soft-clipped"] >= split_num)]
    return df_limit_sd

In [ ]:
def limit_score(df_limit_bed, score=5):
    df_limit_score = df_limit_bed[df_limit_bed["score"] >= score]
    return df_limit_score

In [ ]:
def limit_N(sequence, N_num=5):
    return sequence.count('N') <= N_num

In [ ]:
def limit_simple_repeat(sequence, min_period=2, max_period=4, threshold=0.8):
    seq_upper = sequence.upper()
    for k in range(min_period, max_period + 1):
        pattern = r'(.{' + str(k) + r'})\1+'
        matches = re.finditer(pattern, seq_upper)
        total_repeat_len = 0
        for match in matches:
            total_repeat_len += len(match.group(0))
        if total_repeat_len / len(sequence) >= threshold:
            return True           
    return False

In [ ]:
def limit_chr(low=1, high=12):
    chrom = [f"chr{i}" for i in range(low, high + 1)]
    return chrom

In [ ]:
def extract_sequences(bed_file, genome_file):
    dict_genome = SeqIO.to_dict(SeqIO.parse(genome_file, "fasta"))
    df_bed_length = limit_length(bed_file)
    df_bed_disc_split = limit_split_disc(df_bed_length)
    df_bed_score = limit_score(df_bed_disc_split)
    extracted_data = []
    for _, row in df_bed_score.iterrows():
        chrom = str(row['chrom'])
        if chrom not in limit_chr():
            continue
        start = int(row['start'])
        end = int(row['end'])
        record = dict_genome[chrom]
        seq_length = len(record.seq)
        if start < 0:
            start = 0
        if end > seq_length:
            end = seq_length   
        if start >= end:
            print(f"start >= end: {chrom}:{start}-{end}")
            continue
        sequence = str(record.seq[start:end])
        if not limit_N(sequence):
            continue
        if limit_simple_repeat(sequence):
            continue
        location_str = f"{chrom}:{start}-{end}"
        extracted_data.append([location_str, sequence])
    df_limit_sequence = pd.DataFrame(extracted_data, columns=['location', 'sequence'])
    return df_limit_sequence